## Curso Integral de Econometría 
### Modulo: Econometría espacial
### Tema: Heterogeneidad espacial
### Sesión: 13
### Fecha: 14/11/2026
### Docente/Asesor: Alexis Adonai Morales Alberto

#### **Módulos, submódulos, clases y configuración inicial**

In [2]:
# Instalación del módulo mgwr 

!pip install mgwr

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [spglm]
   -------------------- ------------------- 1/2 [mgwr]
   ---------------------------------------- 2/2 [mgwr]




[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Módulos 

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import geopandas as gpd

# Submódulos y clases 

from libpysal.weights import Queen
from esda.moran import Moran
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW
from pathlib import Path
from scipy.stats import chi2
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# Configuración de la sesión

BASE = Path(
    "E:\Econometrics Data Lab\Curso integral de econometría\Econometría espacial\E_SP_SEP26"
)
RUTAS_BASE = [BASE / 'Delincuencia_Mexico_2024.csv', BASE / "Datos" / "Delincuencia_Mexico_2024.csv"]
RUTA_DATOS = next((p for p in RUTAS_BASE if p.exists()), RUTAS_BASE[0])
MAPAS = [BASE / 'Mapas' / 'Mexico_ent' / '00ent.shp']
RUTA_MAPA = next((p for p in MAPAS if p.exists()), MAPAS[0])
SALIDA = BASE / 'Resultados_S13'
FIG = SALIDA / 'Figuras'
FIG.mkdir(parents=True, exist_ok=True)
NARANJA = '#F04A1D'; TURQUESA = '#08989C'; AZUL = '#003057'
GRIS = '#4D565E'; FONDO = '#F7F7F5'; BLANCO = '#FFFFFF'
DPI = 300
plt.rcParams.update({'figure.facecolor': FONDO, 'axes.facecolor': FONDO,
                     'font.size': 10, 'axes.labelcolor': GRIS, 'xtick.color': GRIS,
                     'ytick.color': GRIS, 'axes.spines.top': False,
                     'axes.spines.right': False})

#### **Lectura de datos**

In [7]:
Datos = pd.read_csv(RUTA_DATOS, dtype={'CVE_ENT': str}, encoding='utf-8-sig')
Datos.columns = Datos.columns.str.strip()
Datos['CVE_ENT'] = Datos['CVE_ENT'].astype(str).str.zfill(2)
requeridas = ['CVE_ENT', 'ENTIDAD', 'Tasa_Incidencia_ENVIPE_2024',
              'Pobreza_2024', 'Gasto_Hogar_2024', 'Desocupacion_2024T2']
if set(requeridas) - set(Datos.columns):
    raise ValueError(f'Faltan columnas: {set(requeridas) - set(Datos.columns)}')
if Datos['CVE_ENT'].duplicated().any() or len(Datos) != 32:
    raise ValueError('Se esperan 32 claves estatales únicas.')
for c in requeridas[2:]:
    Datos[c] = pd.to_numeric(Datos[c], errors='raise')
if Datos[requeridas[2:]].isna().any().any():
    raise ValueError('Existen valores faltantes en las variables principales.')
if (Datos['Tasa_Incidencia_ENVIPE_2024'] <= 0).any():
    raise ValueError('La incidencia debe ser estrictamente positiva para logaritmo.')
Datos = Datos.sort_values('CVE_ENT').reset_index(drop=True)
Datos['Ln_Incidencia'] = np.log(Datos['Tasa_Incidencia_ENVIPE_2024'])
Datos['Gasto_miles'] = Datos['Gasto_Hogar_2024'] / 1000
Datos['Pobreza'] = Datos['Pobreza_2024']
Datos['Desocupacion'] = Datos['Desocupacion_2024T2']
VARS = ['Pobreza', 'Gasto_miles', 'Desocupacion']


#### **Regímenes geográficos elegido antes de observar el ajuste**

In [12]:
CLAVES_NORTE = {'02','03','05','08','10','19','25','26','28','32'}
Datos['Regimen'] = np.where(Datos['CVE_ENT'].isin(CLAVES_NORTE), 'Norte', 'Centro-Sur')
Datos['Norte'] = (Datos['Regimen'] == 'Norte').astype(int)
print('Distribución de regímenes:')
print(Datos.groupby('Regimen').size())
print('\nEstadísticas descriptivas:')
print(Datos[['Ln_Incidencia'] + VARS].describe().round(3))
print(Datos[['Ln_Incidencia', 'Regimen'] + VARS].groupby('Regimen').describe().round(3))
Datos.to_csv(SALIDA / '01_Base_preparada_S13.csv', index=False, encoding='utf-8-sig')

Distribución de regímenes:
Regimen
Centro-Sur    22
Norte         10
dtype: int64

Estadísticas descriptivas:
       Ln_Incidencia  Pobreza  Gasto_miles  Desocupacion
count         32.000   32.000       32.000        32.000
mean          10.333   28.025       46.687         2.541
std            0.247   14.349        8.788         0.757
min            9.986    9.900       27.118         1.040
25%           10.164   16.825       41.192         2.085
50%           10.312   26.300       46.266         2.490
75%           10.478   35.650       52.429         2.883
max           10.905   66.000       66.383         4.040
           Ln_Incidencia                                                \
                   count    mean    std    min     25%     50%     75%   
Regimen                                                                  
Centro-Sur          22.0  10.389  0.267  9.986  10.176  10.396  10.590   
Norte               10.0  10.211  0.143  9.995  10.134  10.229  10.286   

      

#### **OLS homogéneo: línea de referencia**

In [13]:
y = Datos['Ln_Incidencia'].to_numpy(dtype=float)
X0 = sm.add_constant(Datos[VARS], has_constant='add').astype(float)
ols = sm.OLS(y, X0).fit()
ols_hc3 = ols.get_robustcov_results(cov_type='HC3')
print('\n=== OLS homogéneo ===\n', ols.summary())
print('\n=== OLS errores robustos HC3 ===\n', ols_hc3.summary())

VIF = pd.DataFrame({'Regresor': VARS,
                    'VIF': [variance_inflation_factor(X0.to_numpy(), j)
                            for j in range(1, X0.shape[1])]})
print('\nVIF:\n', VIF.to_string(index=False))
VIF.to_csv(SALIDA / '02_VIF.csv', index=False)
LM_BP, p_BP, F_BP, pF_BP = het_breuschpagan(ols.resid, X0)
LM_W, p_W, F_W, pF_W = het_white(ols.resid, X0)
print(f'Breusch-Pagan: LM={LM_BP:.3f}, p={p_BP:.4f}')
print(f'White: LM={LM_W:.3f}, p={p_W:.4f}')


=== OLS homogéneo ===
                             OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.519
Model:                            OLS   Adj. R-squared:                  0.468
Method:                 Least Squares   F-statistic:                     10.08
Date:                Sat, 14 Nov 2026   Prob (F-statistic):           0.000113
Time:                        11:46:01   Log-Likelihood:                 11.536
No. Observations:                  32   AIC:                            -15.07
Df Residuals:                      28   BIC:                            -9.210
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            7.8507 

#### **Modelo de regíemens: intercepto y pobreza variables**

In [14]:
Datos['Norte_x_Pobreza'] = Datos['Norte'] * Datos['Pobreza']
XR = sm.add_constant(Datos[VARS + ['Norte', 'Norte_x_Pobreza']], has_constant='add').astype(float)
regimen = sm.OLS(y, XR).fit()
regimen_hc3 = regimen.get_robustcov_results(cov_type='HC3')
print('\n=== Regímenes espaciales (dos grupos) ===\n', regimen.summary())
print('\n=== Regímenes con HC3 ===\n', regimen_hc3.summary())
print('\nPendiente pobreza Centro-Sur:', regimen.params['Pobreza'])
print('Pendiente pobreza Norte:', regimen.params['Pobreza'] + regimen.params['Norte_x_Pobreza'])


=== Regímenes espaciales (dos grupos) ===
                             OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.569
Model:                            OLS   Adj. R-squared:                  0.486
Method:                 Least Squares   F-statistic:                     6.855
Date:                Sat, 14 Nov 2026   Prob (F-statistic):           0.000336
Time:                        11:50:00   Log-Likelihood:                 13.272
No. Observations:                  32   AIC:                            -14.54
Df Residuals:                      26   BIC:                            -5.750
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------

#### **Chow/Wald: igualdad conjunta de intercepto y pendiente pobreza**

In [15]:
restricciones = 'Norte = 0, Norte_x_Pobreza = 0'
prueba_F = regimen.f_test(restricciones)             # Chow parcial clásico
prueba_robusta = regimen_hc3.wald_test(restricciones, scalar=True)  # F robusta HC3
print('\nChow parcial (clásico):', prueba_F)
print('Wald HC3 para estabilidad entre regímenes:', prueba_robusta)
print('Prueba específica diferencia de pendiente pobreza:',
      regimen_hc3.t_test('Norte_x_Pobreza = 0'))


Chow parcial (clásico): <F test: F=1.489504851564183, p=0.24409984289751047, df_denom=26, df_num=2>
Wald HC3 para estabilidad entre regímenes: <F test: F=1.4181473199918786, p=0.26027989148590613, df_denom=26, df_num=2>
Prueba específica diferencia de pendiente pobreza:                              Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.0083      0.008      1.082      0.289      -0.007       0.024


#### **Comparación descriptiva de OLS y regímenes**

In [16]:
comparacion = pd.DataFrame([
    {'Modelo': 'OLS', 'N': int(ols.nobs), 'k_sin_varianza': int(ols.df_model + 1),
     'R2': ols.rsquared, 'R2_adj': ols.rsquared_adj, 'AIC': ols.aic, 'BIC': ols.bic},
    {'Modelo': 'Regimen (intercepto y pobreza)', 'N': int(regimen.nobs),
     'k_sin_varianza': int(regimen.df_model + 1), 'R2': regimen.rsquared,
     'R2_adj': regimen.rsquared_adj, 'AIC': regimen.aic, 'BIC': regimen.bic},
])
print('\nComparación OLS/regímenes:\n', comparacion.round(4).to_string(index=False))
comparacion.to_csv(SALIDA / '03_Comparacion_OLS_Regimen.csv', index=False)



Comparación OLS/regímenes:
                         Modelo  N  k_sin_varianza     R2  R2_adj      AIC     BIC
                           OLS 32               4 0.5192  0.4677 -15.0728 -9.2098
Regimen (intercepto y pobreza) 32               6 0.5686  0.4857 -14.5440 -5.7496


#### **Coeficientes y errores estándar robustos exportables**

In [18]:
nombres = list(XR.columns)
tabla = pd.DataFrame({'Variable': nombres, 'Coeficiente': regimen.params.to_numpy(),
                      'SE_HC3': regimen_hc3.bse, 't_HC3': regimen_hc3.tvalues,
                      'p_HC3': regimen_hc3.pvalues})
tabla.to_csv(SALIDA / '04_Coeficientes_Regimen.csv', index=False)
Datos['Residuo_OLS'] = np.asarray(ols.resid)
Datos['Residuo_Regimen'] = np.asarray(regimen.resid)
Datos['Prediccion_OLS'] = np.asarray(ols.fittedvalues)
Datos['Prediccion_Regimen'] = np.asarray(regimen.fittedvalues)
Datos.to_csv(SALIDA / '05_Datos_Residuos.csv', index=False, encoding='utf-8-sig')
Datos

,CVE_ENT,ENTIDAD,Anio,Tasa_Incidencia_ENVIPE_2024,Tasa_Incidencia_ENVIPE_2023,Tasa_Victimas_ENVIPE_2024,Tasa_Victimas_ENVIPE_2023,Pobreza_2024,Pobreza_Extrema_2024,Ingreso_Hogar_2024,...,Gasto_miles,Pobreza,Desocupacion,Regimen,Norte,Norte_x_Pobreza,Residuo_OLS,Residuo_Regimen,Prediccion_OLS,Prediccion_Regimen
0,01,Aguascalientes,2024,33960,45262,29473,32798,17.1,0.6,89819,...,53.870,17.1,3.36,Centro-Sur,0,0.0,-0.023292,-0.089507,10.456231,10.522446
1,02,Baja California,2024,25425,31198,20647,23796,9.9,0.4,101187,...,55.269,9.9,2.74,Norte,1,9.9,-0.144955,-0.053138,10.288443,10.196626
2,03,Baja California Sur,2024,21922,25661,18827,21621,10.2,1.2,104728,...,55.939,10.2,2.41,Norte,1,10.2,-0.284828,-0.192578,10.280074,10.187824
3,04,Campeche,2024,22071,31793,18610,22560,36.7,5.8,66236,...,41.574,36.7,2.00,Centro-Sur,0,0.0,-0.266761,-0.286182,10.268781,10.288202
4,05,Coahuila de Zaragoza,2024,22266,24661,20007,19371,12.4,0.8,87652,...,50.501,12.4,3.71,Norte,1,12.4,-0.273166,-0.204327,10.283982,10.215144
5,06,Colima,2024,30839,28522,18882,21206,15.0,1.0,86241,...,52.016,15.0,2.11,Centro-Sur,0,0.0,0.130325,0.036030,10.206210,10.300506
6,07,Chiapas,2024,22366,16038,15576,14139,66.0,27.1,41084,...,27.118,66.0,1.57,Centro-Sur,0,0.0,-0.293338,-0.214989,10.308635,10.230286
7,08,Chihuahua,2024,27140,29090,20601,22565,15.1,2.2,92363,...,46.881,15.1,2.53,Norte,1,15.1,0.131239,0.169313,10.077525,10.039451
8,09,Ciudad de México,2024,54473,52723,30804,32497,19.7,1.8,110685,...,66.383,19.7,4.04,Centro-Sur,0,0.0,-0.116750,-0.105453,11.022210,11.010913
9,10,Durango,2024,28244,18230,20215,16375,27.9,4.3,69589,...,42.456,27.9,2.81,Norte,1,27.9,0.034365,0.020810,10.214272,10.227827


#### **Gráficos (sin mapa)**

In [19]:
fig, ax = plt.subplots(figsize=(9, 6), dpi = 500)
for r, color in [('Norte', NARANJA), ('Centro-Sur', AZUL)]:
    f = Datos['Regimen'] == r
    ax.scatter(Datos.loc[f, 'Pobreza'], Datos.loc[f, 'Ln_Incidencia'],
               color=color, s=55, edgecolor='white', label=f'{r} (n={f.sum()})')
ax.set_xlabel('Pobreza multidimensional (%)')
ax.set_ylabel('Logaritmo de la tasa de incidencia (ENVIPE)')
ax.legend(frameon=False)
ax.set_title('Heterogeneidad regional: distribución de pobreza e incidencia', pad=18)
fig.tight_layout()
fig.savefig(FIG / '01_Dispersion_Regimen.png', dpi=DPI)
fig.savefig(FIG / '01_Dispersion_Regimen.svg')
plt.close(fig)

fig, axs = plt.subplots(1, 2, figsize=(13, 5), dpi = 500)
for ax, col, nombre in zip(axs, ['Residuo_OLS', 'Residuo_Regimen'], ['OLS', 'Regímenes']):
    ax.scatter(Datos['CVE_ENT'], Datos[col], color=NARANJA if nombre == 'OLS' else TURQUESA, s=25)
    ax.axhline(0, color=GRIS, linewidth=1)
    ax.set_title(f'Residuos: {nombre}')
    ax.set_xlabel('CVE_ENT')
    ax.tick_params(axis='x', labelrotation=90, labelsize=7)
axs[0].set_ylabel('Residuo en log incidencia')
fig.tight_layout()
fig.savefig(FIG / '02_Residuos_por_entidad.png', dpi=DPI)
fig.savefig(FIG / '02_Residuos_por_entidad.svg')
plt.close(fig)


#### **Catografía y matriz W**

In [20]:
Mex = None
w = None
if RUTA_MAPA.exists():
    try:
        mapa = gpd.read_file(RUTA_MAPA)
        mapa.columns = mapa.columns.str.strip()
        mapa['CVE_ENT'] = mapa['CVE_ENT'].astype(str).str.zfill(2)
        if mapa.crs is None:
            raise ValueError('El shapefile no contiene CRS; comprueba el .prj')
        if mapa.geometry.isna().any() or mapa.geometry.is_empty.any():
            raise ValueError('Existen geometrías vacías')
        if not mapa.geometry.is_valid.all():
            raise ValueError('Corrige geometrías inválidas antes de continuar')
        mapa = mapa[['CVE_ENT', 'geometry']].dissolve(by='CVE_ENT', as_index=False)
        Mex = mapa.merge(Datos, on='CVE_ENT', validate='one_to_one').sort_values('CVE_ENT').reset_index(drop=True)
        if len(Mex) != 32:
            raise ValueError('La unión cartográfica no conserva las 32 entidades')
        # CRS proyectado en metros para GWR y las distancias geográficas.
        Mex = Mex.to_crs(epsg=6372)
        w = Queen.from_dataframe(Mex, use_index=True)
        if w.islands:
            raise ValueError(f'Hay entidades sin vecinos en contigüidad reina: {w.islands}')
        w.transform = 'R'
        for modelo, residuo in [('OLS', 'Residuo_OLS'), ('Regimen', 'Residuo_Regimen')]:
            moran = Moran(Mex[residuo].to_numpy(), w, permutations=9999)
            print(f'Moran residual {modelo}: I={moran.I:.4f}, p_sim unilateral={moran.p_sim:.4f}; interpretar dirección del contraste')
            fig, ax = plt.subplots(figsize=(11, 7), dpi = 500)
            Mex.plot(column=residuo, ax=ax, cmap='RdBu_r', legend=True,
                     legend_kwds={'shrink': .65}, edgecolor=BLANCO, linewidth=.65)
            ax.set_axis_off()
            ax.set_title(f'Residuos de {modelo} | Moran I={moran.I:.3f}; p_sim={moran.p_sim:.3f}')
            fig.tight_layout()
            fig.savefig(FIG / f'03_Mapa_Residuos_{modelo}.png', dpi=DPI)
            fig.savefig(FIG / f'03_Mapa_Residuos_{modelo}.svg')
            plt.close(fig)
        Mex[['CVE_ENT', 'Regimen','Pobreza', 'Residuo_OLS', 'Residuo_Regimen', 'geometry']].to_file(
            SALIDA / 'Residuos_y_Regimen.gpkg', driver='GPKG')
    except ImportError as err:
        print('Cartografía presente pero falta instalar dependencia geográfica:', err)
else:
    print('\nNo se encontró 00ent.shp. Se omiten cartografía, Moran, expansión espacial y GWR.')


Moran residual OLS: I=-0.0648, p_sim unilateral=0.4074; interpretar dirección del contraste
Moran residual Regimen: I=-0.0043, p_sim unilateral=0.3966; interpretar dirección del contraste


#### **Expansión espacial: pendientes que varíuan con las coordenadas**

In [21]:
expansion = None
if Mex is not None:
    centros = Mex.geometry.centroid
    Mex['Coord_X'] = (centros.x - centros.x.mean()) / centros.x.std(ddof=0)
    Mex['Coord_Y'] = (centros.y - centros.y.mean()) / centros.y.std(ddof=0)
    Mex['Pobreza_X'] = Mex['Pobreza'] * Mex['Coord_X']
    Mex['Pobreza_Y'] = Mex['Pobreza'] * Mex['Coord_Y']
    # Controlar tendencias espaciales de intercepto para separar variaciones
    # espaciales de la pendiente de pobreza de una tendencia de nivel regional.
    XX = sm.add_constant(Mex[VARS + ['Coord_X', 'Coord_Y', 'Pobreza_X', 'Pobreza_Y']], has_constant='add')
    expansion = sm.OLS(Mex['Ln_Incidencia'], XX).fit()
    expansion_hc3 = expansion.get_robustcov_results(cov_type='HC3')
    print('\n=== Expansión espacial ===\n', expansion.summary())
    print('Prueba robusta de variación espacial de pendiente:',
          expansion_hc3.wald_test('Pobreza_X = 0, Pobreza_Y = 0', scalar=True))
    Mex['Beta_Pobreza_Expansion'] = (expansion.params['Pobreza']
                                    + expansion.params['Pobreza_X'] * Mex['Coord_X']
                                    + expansion.params['Pobreza_Y'] * Mex['Coord_Y'])
    Mex['Residuo_Expansion'] = expansion.resid.to_numpy()
    fig, ax = plt.subplots(figsize=(11, 7), dpi = 500)
    Mex.plot(column='Beta_Pobreza_Expansion', ax=ax, cmap='coolwarm',
             legend=True, edgecolor=BLANCO, linewidth=.6)
    ax.set_axis_off(); ax.set_title('Pendiente condicional de pobreza | expansión espacial')
    fig.tight_layout()
    fig.savefig(FIG / '04_Beta_Pobreza_Expansion.png', dpi=DPI)
    fig.savefig(FIG / '04_Beta_Pobreza_Expansion.svg')
    plt.close(fig)
    Mex[['CVE_ENT', 'Beta_Pobreza_Expansion', 'geometry']].to_file(
        SALIDA / 'Expansion_Espacial.gpkg', driver='GPKG')
    pd.DataFrame({'Modelo': ['OLS','Regimen','Expansion'],
                  'AIC': [ols.aic, regimen.aic, expansion.aic],
                  'BIC': [ols.bic, regimen.bic, expansion.bic]}).to_csv(
                      SALIDA / '06_Comparacion_OLS_Regimen_Expansion.csv', index=False)



=== Expansión espacial ===
                             OLS Regression Results                            
Dep. Variable:          Ln_Incidencia   R-squared:                       0.603
Model:                            OLS   Adj. R-squared:                  0.488
Method:                 Least Squares   F-statistic:                     5.218
Date:                Sat, 14 Nov 2026   Prob (F-statistic):            0.00102
Time:                        12:15:31   Log-Likelihood:                 14.619
No. Observations:                  32   AIC:                            -13.24
Df Residuals:                      24   BIC:                            -1.512
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            8.

#### **Estimación de GWR**

In [22]:
if Mex is not None:
    try:
        coords = np.column_stack([Mex.geometry.centroid.x.to_numpy(),
                                  Mex.geometry.centroid.y.to_numpy()])
        yg = Mex[['Ln_Incidencia']].to_numpy(dtype=float)
        xg = Mex[['Pobreza', 'Gasto_miles']].to_numpy(dtype=float)
        selector = Sel_BW(coords, yg, xg, fixed=False, kernel='bisquare')
        vecinos_gwr = int(selector.search(search_method='golden_section', bw_min=18,
                                          bw_max=len(Mex)))
        print('\nAncho de banda adaptativo GWR (número de vecinos):', vecinos_gwr)
        gwr = GWR(coords, yg, xg, bw=vecinos_gwr,
                  fixed=False, kernel='bisquare', constant=True).fit()
        print(gwr.summary())
        Mex['Beta_Pobreza_GWR'] = gwr.params[:, 1]
        Mex['Beta_Gasto_GWR'] = gwr.params[:, 2]
        Mex['Residuo_GWR'] = yg.ravel() - gwr.predy.ravel()
        print('AICc GWR:', gwr.aicc)
        for c in ['Beta_Pobreza_GWR','Beta_Gasto_GWR']:
            fig, ax = plt.subplots(figsize=(11, 7), dpi = 500)
            Mex.plot(column=c, cmap='coolwarm', ax=ax, legend=True,
                     edgecolor=BLANCO, linewidth=.7)
            ax.set_axis_off(); ax.set_title(f'{c}: modelo GWR exploratorio')
            fig.tight_layout()
            fig.savefig(FIG / f'05_{c}.png', dpi=DPI)
            fig.savefig(FIG / f'05_{c}.svg')
            plt.close(fig)
        Mex[['CVE_ENT','Beta_Pobreza_GWR','Beta_Gasto_GWR','geometry']].to_file(
            SALIDA / 'GWR_Coeficientes.gpkg', driver='GPKG')
    except ImportError:
        print('GWR opcional no ejecutado: instala mgwr.')
    except Exception as err:
        warnings.warn(f'GWR no pudo estimarse con esta muestra/configuración: {err}')



Ancho de banda adaptativo GWR (número de vecinos): 31
Model type                                                         Gaussian
Number of observations:                                                  32
Number of covariates:                                                     3

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                              1.074
Log-likelihood:                                                       8.906
AIC:                                                                -11.811
AICc:                                                                -8.330
BIC:                                                                -99.432
R2:                                                                   0.433
Adj. R2:                                                              0.394

Variable                              Est.         SE  t(Est/SE)    p-value
-----

#### **Intepretación final asistida**

In [23]:
print('\n=== RESULTADOS PRINCIPALES ===')
print('OLS R2 ajustado:', round(ols.rsquared_adj, 4))
print('Regímenes R2 ajustado:', round(regimen.rsquared_adj, 4))
print('Chow clásico p:', float(prueba_F.pvalue))
print('Wald HC3 p:', float(prueba_robusta.pvalue))
print('Nota: regímenes definidos ex ante; significancia no implica causalidad.')
print('Archivos guardados:', SALIDA.resolve())


=== RESULTADOS PRINCIPALES ===
OLS R2 ajustado: 0.4677
Regímenes R2 ajustado: 0.4857
Chow clásico p: 0.24409984289751047
Wald HC3 p: 0.26027989148590613
Nota: regímenes definidos ex ante; significancia no implica causalidad.
Archivos guardados: E:\Econometrics Data Lab\Curso integral de econometría\Econometría espacial\E_SP_SEP26\Resultados_S13
